In [1]:
import time 
import multiprocessing as mp 

import cv2
import pandas as pd
from tqdm import tqdm 

In [3]:
src = '/home/amos/datasets/CineFace/encoding_test.csv'
df = pd.read_csv(src, index_col=0)
df.head()

,x1,y1,x2,y2,right_eye_x,right_eye_y,left_eye_x,left_eye_y,nose_x,nose_y,...,frame_num,face_num,img_width,img_height,filename,series_id,distance_from_center,pct_of_frame,encoding,filepath
0,1661,454,2551,1592,1871,934,2285,943,2050,1175,...,408,0,3840,2160,A.Murder.at.the.End.of.the.World.S01E06.Crime....,15227418,32.86,0.1221,[-1.95243880e-01 5.42654619e-02 7.73123279e-...,/home/amos/media/tv/a_murder_at_the_end_of_the...
1,1908,1079,1994,1194,1936,1120,1976,1120,1959,1141,...,456,0,3840,2160,A.Murder.at.the.End.of.the.World.S01E06.Crime....,15227418,34.55,0.0012,[-0.08516457 0.11344527 0.04984289 -0.018541...,/home/amos/media/tv/a_murder_at_the_end_of_the...
2,2497,1022,2582,1133,2516,1066,2555,1063,2535,1085,...,456,1,3840,2160,A.Murder.at.the.End.of.the.World.S01E06.Crime....,15227418,29.31,0.0011,[-0.20053279 0.0865107 0.07712638 -0.154692...,/home/amos/media/tv/a_murder_at_the_end_of_the...
3,1588,806,1676,932,1613,849,1653,856,1630,875,...,456,2,3840,2160,A.Murder.at.the.End.of.the.World.S01E06.Crime....,15227418,34.60,0.0013,[-0.08482741 0.17685468 0.08484279 -0.058745...,/home/amos/media/tv/a_murder_at_the_end_of_the...
4,2046,1165,2107,1245,2056,1197,2079,1198,2062,1212,...,456,3,3840,2160,A.Murder.at.the.End.of.the.World.S01E06.Crime....,15227418,29.77,0.0006,[-0.14753576 0.06418005 0.12581752 -0.101637...,/home/amos/media/tv/a_murder_at_the_end_of_the...


In [4]:
cap = cv2.VideoCapture(df.at[0, 'filepath'])
framecount = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

### Extract Faces Method 1

In [8]:
def extract_face_old(row,
                 cap):
    cap.set(cv2.CAP_PROP_POS_FRAMES, row['frame_num'])
    ret, frame = cap.read()
    if not ret or frame is None:
        return None

    x1, y1, x2, y2 = row['x1'], row['y1'], row['x2'], row['y2']
    face = frame[y1:y2, x1:x2]
    return face

In [ ]:
faces = []
for idx, row in tqdm(df.iterrows(), total=df.shape[0]):
    face = extract_face_old(row, cap)
    rgb = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
    faces.append(cv2.resize(rgb, (150, 150), interpolation=cv2.INTER_AREA))

### Extract Faces Method 2

In [5]:
def extract_face(row,
                 frame):
    x1, y1, x2, y2 = row['x1'], row['y1'], row['x2'], row['y2']
    face = frame[y1:y2, x1:x2]
    return face

In [6]:
def process_image(face):
    rgb = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
    return cv2.resize(rgb, (150, 150), interpolation=cv2.INTER_AREA)

In [15]:
t = time.time()
frame_nums = df['frame_num'].tolist()
faces = []
for frame_num in tqdm(frame_nums[:int(len(frame_nums)/3)]):
    ret, frame = cap.read()
    temp = df[df['frame_num'] == frame_num]
    for idx, row in temp.iterrows():
        face = extract_face(row, frame)
        faces.append(face)
print(time.time() - t)

100%|██████████| 868/868 [00:22<00:00, 38.38it/s]

22.617537021636963


#### Sequential

In [16]:
t = time.time()
for face in tqdm(faces):
    process_image(face)
print(time.time() - t)

100%|██████████| 1972/1972 [00:01<00:00, 1032.80it/s]

1.9110214710235596


#### Parallel

Parallel processing is actually slower because of the increased overhead. Any performance gain would require a large sample, which takes up too much memory. PARALLEL PROCESSING IS NOT AN OPTION AND IS RULED OUT.

In [17]:
t = time.time()
with mp.Pool(12) as p:
    processed = p.map(process_image, faces)
print(time.time() - t)    

10.129539012908936


In [16]:
len(faces)

0

In [20]:
df.shape

(2605, 25)

In [21]:
df.at[0, 'filepath']

'/home/amos/media/tv/a_murder_at_the_end_of_the_world/A.Murder.at.the.End.of.the.World.S01E06.Crime.Seen.2160p.HULU.WEB-DL.DDP5.1.HEVC-CMRG.mkv'